# Formula 1 Race Outcome Prediction

## Objective

This notebook builds pre-race predictions for every driver entered in a Formula 1 Grand Prix. It addresses two related tasks using shared data cleaning, exploratory data analysis, and feature engineering, with separate final models as appropriate:

1. Estimate each driver's probability of finishing on the podium (`positionOrder <= 3`) and in the points (`positionOrder <= 10`).
2. Predict each driver's finishing position as a numeric score (`posicion_pred`), which will be converted to an ordering within each race (lower scores rank ahead).

The target is `positionOrder`, which is defined for all classified results, including drivers who do not finish. The prediction unit is one driver-result row from `results.csv`; include Grand Prix races only, not sprint events.

## Data and prediction-time rules

Use the Kaggle **Formula 1 World Championship (1950–2024)** dataset, with its CSV files located in the same folder as this notebook. Join related tables using their appropriate keys and document the data preparation. Exclude any records from 2025 if present.

Assume predictions are made after qualifying and before the Grand Prix. Features may include the starting grid and qualifying times from that weekend, that weekend's sprint result when available, and information from earlier races. Historical form features may use earlier races from the 2024 season as they would have been known at the time of each prediction.

Do not use information from the race being predicted, including its finishing position, points, laps, elapsed time, fastest lap, status, lap-time records, or pit-stop records. Driver, constructor, and constructor-result standings for a race reflect the championship after that race; when using standings, align them from the previous race so that no post-race information leaks into features.

## Training, validation, and test

Use seasons through 2023 for training and reserve the complete 2024 season as the test set. No 2024 data may be used to fit models, select features, choose hyperparameters, or otherwise tune the approach. Choose and justify the historical training window. Perform validation only within the pre-2024 training data, preferably with a time-based split (for example, 2023 as validation).

Set `SEED = 2502` and make the notebook reproducible. It must run from top to bottom using the CSV files in the notebook's folder.

## Evaluation and baseline

Evaluate the probability predictions on all 2024 test rows using log loss separately for `p_podio` and `p_top10`, and report their mean. Ensure `p_podio <= p_top10` for every row.

Evaluate predicted positions by converting `posicion_pred` to an ordering within each race. Report the mean per-race Spearman correlation with the actual finishing order; use mean per-race MAE between the predicted ordering and actual order as the tie-break metric.

Include a reference baseline based only on the starting grid, and compare the developed models against it.

## Deliverable

The final notebook cell must export one CSV row per 2024 test result with exactly these columns: `resultId`, `raceId`, `driverId`, `p_podio`, `p_top10`, and `posicion_pred`.

In [2]:
# import main libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [34]:
circuits_df = pd.read_csv("data/circuits.csv")
circuits_df.head(10)

,circuitId,circuitRef,name,location,country,lat,lng,alt,url
0,1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.84970,144.96800,10,http://en.wikipedia.org/wiki/Melbourne_Grand_P...
1,2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.73800,18,http://en.wikipedia.org/wiki/Sepang_Internatio...
2,3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.03250,50.51060,7,http://en.wikipedia.org/wiki/Bahrain_Internati...
3,4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57000,2.26111,109,http://en.wikipedia.org/wiki/Circuit_de_Barcel...
4,5,istanbul,Istanbul Park,Istanbul,Turkey,40.95170,29.40500,130,http://en.wikipedia.org/wiki/Istanbul_Park
5,6,monaco,Circuit de Monaco,Monte-Carlo,Monaco,43.73470,7.42056,7,http://en.wikipedia.org/wiki/Circuit_de_Monaco
6,7,villeneuve,Circuit Gilles Villeneuve,Montreal,Canada,45.50000,-73.52280,13,http://en.wikipedia.org/wiki/Circuit_Gilles_Vi...
7,8,magny_cours,Circuit de Nevers Magny-Cours,Magny Cours,France,46.86420,3.16361,228,http://en.wikipedia.org/wiki/Circuit_de_Nevers...
8,9,silverstone,Silverstone Circuit,Silverstone,UK,52.07860,-1.01694,153,http://en.wikipedia.org/wiki/Silverstone_Circuit
9,10,hockenheimring,Hockenheimring,Hockenheim,Germany,49.32780,8.56583,103,http://en.wikipedia.org/wiki/Hockenheimring


In [4]:
circuits_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   circuitId   77 non-null     int64  
 1   circuitRef  77 non-null     str    
 2   name        77 non-null     str    
 3   location    77 non-null     str    
 4   country     77 non-null     str    
 5   lat         77 non-null     float64
 6   lng         77 non-null     float64
 7   alt         77 non-null     int64  
 8   url         77 non-null     str    
dtypes: float64(2), int64(2), str(5)
memory usage: 12.5 KB


The `circuits_df` dataset has **9 variables** and **no missing values**. It describes each circuit with its name, location, coordinates, and altitude. The races reviewed so far identify circuits by `circuitId`; joining on that field will add the circuit details to each race in `races_df`.

In [35]:
constructor_results_df = pd.read_csv("data/constructor_results.csv")
constructor_results_df.head(10)

,constructorResultsId,raceId,constructorId,points,status
0,1,18,1,14.0,\N
1,2,18,2,8.0,\N
2,3,18,3,9.0,\N
3,4,18,4,5.0,\N
4,5,18,5,2.0,\N
5,6,18,6,1.0,\N
6,7,18,7,0.0,\N
7,8,18,8,0.0,\N
8,9,18,9,0.0,\N
9,10,18,10,0.0,\N


In [6]:
constructor_results_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12625 entries, 0 to 12624
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   constructorResultsId  12625 non-null  int64  
 1   raceId                12625 non-null  int64  
 2   constructorId         12625 non-null  int64  
 3   points                12625 non-null  float64
 4   status                12625 non-null  str    
dtypes: float64(1), int64(3), str(1)
memory usage: 517.9 KB


In [36]:
constructor_results_df['status'].value_counts().sort_values(ascending=False).head()

status
\N    12608
D        17
Name: count, dtype: int64

The `constructor_results_df` dataset has **5 variables** and **missing values** in status that have been turned into a '\N' value. It records the points and status awarded to each constructor in a race. Its `raceId` and `constructorId` fields identify both the event and constructor, so they can be used to connect these records to `races_df` and `constructors_df`.

In [37]:
constructor_standings_df = pd.read_csv("data/constructor_standings.csv")
constructor_standings_df.head(10)

,constructorStandingsId,raceId,constructorId,points,position,positionText,wins
0,1,18,1,14.0,1,1,1
1,2,18,2,8.0,3,3,0
2,3,18,3,9.0,2,2,0
3,4,18,4,5.0,4,4,0
4,5,18,5,2.0,5,5,0
5,6,18,6,1.0,6,6,0
6,7,19,1,24.0,1,1,1
7,8,19,2,19.0,2,2,0
8,9,19,3,9.0,4,4,0
9,10,19,4,6.0,5,5,0


In [8]:
constructor_standings_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13391 entries, 0 to 13390
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   constructorStandingsId  13391 non-null  int64  
 1   raceId                  13391 non-null  int64  
 2   constructorId           13391 non-null  int64  
 3   points                  13391 non-null  float64
 4   position                13391 non-null  int64  
 5   positionText            13391 non-null  str    
 6   wins                    13391 non-null  int64  
dtypes: float64(1), int64(5), str(1)
memory usage: 749.3 KB


The `constructor_standings_df` dataset has **7 variables** and **no missing values**. It follows each constructor's championship points, position, and wins as the season progresses. Match on `raceId` and `constructorId` to relate a standing to that constructor's race-level entry in `constructor_results_df`; `constructorId` also connects both datasets to the constructor details in `constructors_df`.

In [38]:
constructors_df = pd.read_csv("data/constructors.csv")
constructors_df.head(10)

,constructorId,constructorRef,name,nationality,url
0,1,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren
1,2,bmw_sauber,BMW Sauber,German,http://en.wikipedia.org/wiki/BMW_Sauber
2,3,williams,Williams,British,http://en.wikipedia.org/wiki/Williams_Grand_Pr...
3,4,renault,Renault,French,http://en.wikipedia.org/wiki/Renault_in_Formul...
4,5,toro_rosso,Toro Rosso,Italian,http://en.wikipedia.org/wiki/Scuderia_Toro_Rosso
5,6,ferrari,Ferrari,Italian,http://en.wikipedia.org/wiki/Scuderia_Ferrari
6,7,toyota,Toyota,Japanese,http://en.wikipedia.org/wiki/Toyota_Racing
7,8,super_aguri,Super Aguri,Japanese,http://en.wikipedia.org/wiki/Super_Aguri_F1
8,9,red_bull,Red Bull,Austrian,http://en.wikipedia.org/wiki/Red_Bull_Racing
9,10,force_india,Force India,Indian,http://en.wikipedia.org/wiki/Racing_Point_Forc...


In [10]:
constructors_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 212 entries, 0 to 211
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   constructorId   212 non-null    int64
 1   constructorRef  212 non-null    str  
 2   name            212 non-null    str  
 3   nationality     212 non-null    str  
 4   url             212 non-null    str  
dtypes: int64(1), str(4)
memory usage: 22.2 KB


The `constructors_df` dataset has **5 variables** and **no missing values**. It provides the names and nationalities behind the constructor IDs used elsewhere. In particular, `constructorId` lets us add these details to the race-level points in `constructor_results_df` and the cumulative championship records in `constructor_standings_df`.

In [39]:
driver_standings_df = pd.read_csv("data/driver_standings.csv")
driver_standings_df.head(10)

,driverStandingsId,raceId,driverId,points,position,positionText,wins
0,1,18,1,10.0,1,1,1
1,2,18,2,8.0,2,2,0
2,3,18,3,6.0,3,3,0
3,4,18,4,5.0,4,4,0
4,5,18,5,4.0,5,5,0
5,6,18,6,3.0,6,6,0
6,7,18,7,2.0,7,7,0
7,8,18,8,1.0,8,8,0
8,9,19,1,14.0,1,1,1
9,10,19,2,11.0,3,3,0


In [12]:
driver_standings_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34863 entries, 0 to 34862
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   driverStandingsId  34863 non-null  int64  
 1   raceId             34863 non-null  int64  
 2   driverId           34863 non-null  int64  
 3   points             34863 non-null  float64
 4   position           34863 non-null  int64  
 5   positionText       34863 non-null  str    
 6   wins               34863 non-null  int64  
dtypes: float64(1), int64(5), str(1)
memory usage: 1.9 MB


The `driver_standings_df` dataset has **7 variables** and **no missing values**. It tracks each driver's championship points, position, and wins after each race. The `raceId` field identifies when each standing was recorded, while `driverId` identifies the driver; together, these keys will let us connect standings to the race calendar in `races_df` and the driver details in `drivers_df`.

In [40]:
drivers_df = pd.read_csv("data/drivers.csv")
drivers_df.head(10)

,driverId,driverRef,number,code,forename,surname,dob,nationality,url
0,1,hamilton,44,HAM,Lewis,Hamilton,1985-01-07,British,http://en.wikipedia.org/wiki/Lewis_Hamilton
1,2,heidfeld,\N,HEI,Nick,Heidfeld,1977-05-10,German,http://en.wikipedia.org/wiki/Nick_Heidfeld
2,3,rosberg,6,ROS,Nico,Rosberg,1985-06-27,German,http://en.wikipedia.org/wiki/Nico_Rosberg
3,4,alonso,14,ALO,Fernando,Alonso,1981-07-29,Spanish,http://en.wikipedia.org/wiki/Fernando_Alonso
4,5,kovalainen,\N,KOV,Heikki,Kovalainen,1981-10-19,Finnish,http://en.wikipedia.org/wiki/Heikki_Kovalainen
5,6,nakajima,\N,NAK,Kazuki,Nakajima,1985-01-11,Japanese,http://en.wikipedia.org/wiki/Kazuki_Nakajima
6,7,bourdais,\N,BOU,Sébastien,Bourdais,1979-02-28,French,http://en.wikipedia.org/wiki/S%C3%A9bastien_Bo...
7,8,raikkonen,7,RAI,Kimi,Räikkönen,1979-10-17,Finnish,http://en.wikipedia.org/wiki/Kimi_R%C3%A4ikk%C...
8,9,kubica,88,KUB,Robert,Kubica,1984-12-07,Polish,http://en.wikipedia.org/wiki/Robert_Kubica
9,10,glock,\N,GLO,Timo,Glock,1982-03-18,German,http://en.wikipedia.org/wiki/Timo_Glock


In [14]:
drivers_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 861 entries, 0 to 860
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   driverId     861 non-null    int64
 1   driverRef    861 non-null    str  
 2   number       861 non-null    str  
 3   code         861 non-null    str  
 4   forename     861 non-null    str  
 5   surname      861 non-null    str  
 6   dob          861 non-null    str  
 7   nationality  861 non-null    str  
 8   url          861 non-null    str  
dtypes: int64(1), str(8)
memory usage: 132.5 KB


The `drivers_df` dataset has **9 variables** and **no missing values**. It contains each driver's identity and biographical details. The `driverId` key connects these details to the championship history in `driver_standings_df` and to individual race, lap, pit-stop, qualifying, and sprint records.

In [41]:
lap_times_df = pd.read_csv("data/lap_times.csv")
lap_times_df.head(10)

,raceId,driverId,lap,position,time,milliseconds
0,841,20,1,1,1:38.109,98109
1,841,20,2,1,1:33.006,93006
2,841,20,3,1,1:32.713,92713
3,841,20,4,1,1:32.803,92803
4,841,20,5,1,1:32.342,92342
5,841,20,6,1,1:32.605,92605
6,841,20,7,1,1:32.502,92502
7,841,20,8,1,1:32.537,92537
8,841,20,9,1,1:33.240,93240
9,841,20,10,1,1:32.572,92572


In [16]:
lap_times_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 589081 entries, 0 to 589080
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   raceId        589081 non-null  int64
 1   driverId      589081 non-null  int64
 2   lap           589081 non-null  int64
 3   position      589081 non-null  int64
 4   time          589081 non-null  str  
 5   milliseconds  589081 non-null  int64
dtypes: int64(5), str(1)
memory usage: 31.5 MB


The `lap_times_df` dataset has **6 variables** and **no missing values**. Each record describes a driver's position and time for one lap, so there may be many rows for the same driver in the same race. Use `raceId` and `driverId` together to identify that driver-event pair and relate lap records to `driver_standings_df` or other race-level datasets.

In [42]:
pit_stops_df = pd.read_csv("data/pit_stops.csv")
pit_stops_df.head(10)

,raceId,driverId,stop,lap,time,duration,milliseconds
0,841,153,1,1,17:05:23,26.898,26898
1,841,30,1,1,17:05:52,25.021,25021
2,841,17,1,11,17:20:48,23.426,23426
3,841,4,1,12,17:22:34,23.251,23251
4,841,13,1,13,17:24:10,23.842,23842
5,841,22,1,13,17:24:29,23.643,23643
6,841,20,1,14,17:25:17,22.603,22603
7,841,814,1,14,17:26:03,24.863,24863
8,841,816,1,14,17:26:50,25.259,25259
9,841,67,1,15,17:27:34,25.342,25342


In [18]:
pit_stops_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11371 entries, 0 to 11370
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   raceId        11371 non-null  int64
 1   driverId      11371 non-null  int64
 2   stop          11371 non-null  int64
 3   lap           11371 non-null  int64
 4   time          11371 non-null  str  
 5   duration      11371 non-null  str  
 6   milliseconds  11371 non-null  int64
dtypes: int64(5), str(2)
memory usage: 778.9 KB


The `pit_stops_df` dataset has **7 variables** and **no missing values**. It records each stop's sequence, lap, and duration; a driver can therefore have several entries in one race. The shared `raceId` and `driverId` fields connect those stops to the same driver's lap-by-lap records in `lap_times_df` and championship record in `driver_standings_df`.

In [43]:
qualifying_df = pd.read_csv("data/qualifying.csv")
qualifying_df.head(10)

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3
0,1,18,1,1,22,1,1:26.572,1:25.187,1:26.714
1,2,18,9,2,4,2,1:26.103,1:25.315,1:26.869
2,3,18,5,1,23,3,1:25.664,1:25.452,1:27.079
3,4,18,13,6,2,4,1:25.994,1:25.691,1:27.178
4,5,18,2,2,3,5,1:25.960,1:25.518,1:27.236
5,6,18,15,7,11,6,1:26.427,1:26.101,1:28.527
6,7,18,3,3,7,7,1:26.295,1:26.059,1:28.687
7,8,18,14,9,9,8,1:26.381,1:26.063,1:29.041
8,9,18,10,7,12,9,1:26.919,1:26.164,1:29.593
9,10,18,20,5,15,10,1:26.702,1:25.842,\N


In [20]:
qualifying_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10494 entries, 0 to 10493
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   qualifyId      10494 non-null  int64
 1   raceId         10494 non-null  int64
 2   driverId       10494 non-null  int64
 3   constructorId  10494 non-null  int64
 4   number         10494 non-null  int64
 5   position       10494 non-null  int64
 6   q1             10494 non-null  str  
 7   q2             10472 non-null  str  
 8   q3             10448 non-null  str  
dtypes: int64(6), str(3)
memory usage: 918.0 KB


The `qualifying_df` dataset has **9 variables** and **68 missing values**. All missing values occur in `q2` (22) and `q3` (46), which is expected when a driver does not progress to a later qualifying session or set a time there. The `raceId` and `driverId` fields identify the event and driver, allowing qualifying pace to be compared with the driver's other race records; `constructorId` connects each entry to team details in `constructors_df`.

In [44]:
races_df = pd.read_csv("data/races.csv")
races_df.head(10)

,raceId,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
0,1,2009,1,1,Australian Grand Prix,2009-03-29,06:00:00,http://en.wikipedia.org/wiki/2009_Australian_G...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
1,2,2009,2,2,Malaysian Grand Prix,2009-04-05,09:00:00,http://en.wikipedia.org/wiki/2009_Malaysian_Gr...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
2,3,2009,3,17,Chinese Grand Prix,2009-04-19,07:00:00,http://en.wikipedia.org/wiki/2009_Chinese_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
3,4,2009,4,3,Bahrain Grand Prix,2009-04-26,12:00:00,http://en.wikipedia.org/wiki/2009_Bahrain_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
4,5,2009,5,4,Spanish Grand Prix,2009-05-10,12:00:00,http://en.wikipedia.org/wiki/2009_Spanish_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
5,6,2009,6,6,Monaco Grand Prix,2009-05-24,12:00:00,http://en.wikipedia.org/wiki/2009_Monaco_Grand...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
6,7,2009,7,5,Turkish Grand Prix,2009-06-07,12:00:00,http://en.wikipedia.org/wiki/2009_Turkish_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
7,8,2009,8,9,British Grand Prix,2009-06-21,12:00:00,http://en.wikipedia.org/wiki/2009_British_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
8,9,2009,9,20,German Grand Prix,2009-07-12,12:00:00,http://en.wikipedia.org/wiki/2009_German_Grand...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
9,10,2009,10,11,Hungarian Grand Prix,2009-07-26,12:00:00,http://en.wikipedia.org/wiki/2009_Hungarian_Gr...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N


In [22]:
races_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1125 entries, 0 to 1124
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   raceId       1125 non-null   int64
 1   year         1125 non-null   int64
 2   round        1125 non-null   int64
 3   circuitId    1125 non-null   int64
 4   name         1125 non-null   str  
 5   date         1125 non-null   str  
 6   time         1125 non-null   str  
 7   url          1125 non-null   str  
 8   fp1_date     1125 non-null   str  
 9   fp1_time     1125 non-null   str  
 10  fp2_date     1125 non-null   str  
 11  fp2_time     1125 non-null   str  
 12  fp3_date     1125 non-null   str  
 13  fp3_time     1125 non-null   str  
 14  quali_date   1125 non-null   str  
 15  quali_time   1125 non-null   str  
 16  sprint_date  1125 non-null   str  
 17  sprint_time  1125 non-null   str  
dtypes: int64(4), str(14)
memory usage: 279.1 KB


In [33]:
races_df['fp1_date'].value_counts().sort_values(ascending=False).head()

fp1_date
\N            1035
2021-04-16       1
2022-03-18       1
2021-03-26       1
2021-11-19       1
Name: count, dtype: int64

The `races_df` dataset has **18 variables** and supposedly **no missing values**, but `fp...`, `quali_...` and `sprint_...` variables have values = '\N' indicating real missing values. It provides the calendar context for each Grand Prix, including its date, round, circuit, and session schedule. The `circuitId` field connects a race to its venue in `circuits_df`; `raceId` is the event key shared by datasets such as `qualifying_df`, `pit_stops_df`, `driver_standings_df`, and `constructor_results_df`.

In [45]:
results_df = pd.read_csv("data/results.csv")
results_df.head(10)

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1
5,6,18,6,3,8,13,6,6,6,3.0,57,\N,\N,50,14,1:29.639,212.974,11
6,7,18,7,5,14,17,7,7,7,2.0,55,\N,\N,54,8,1:29.534,213.224,5
7,8,18,8,6,1,15,8,8,8,1.0,53,\N,\N,20,4,1:27.903,217.180,5
8,9,18,9,2,4,2,\N,R,9,0.0,47,\N,\N,15,9,1:28.753,215.100,4
9,10,18,10,7,12,18,\N,R,10,0.0,43,\N,\N,23,13,1:29.558,213.166,3


In [24]:
results_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26759 entries, 0 to 26758
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   resultId         26759 non-null  int64  
 1   raceId           26759 non-null  int64  
 2   driverId         26759 non-null  int64  
 3   constructorId    26759 non-null  int64  
 4   number           26759 non-null  str    
 5   grid             26759 non-null  int64  
 6   position         26759 non-null  str    
 7   positionText     26759 non-null  str    
 8   positionOrder    26759 non-null  int64  
 9   points           26759 non-null  float64
 10  laps             26759 non-null  int64  
 11  time             26759 non-null  str    
 12  milliseconds     26759 non-null  str    
 13  fastestLap       26759 non-null  str    
 14  rank             26759 non-null  str    
 15  fastestLapTime   26759 non-null  str    
 16  fastestLapSpeed  26759 non-null  str    
 17  statusId         26759 

The `results_df` dataset has **18 variables** and **missing values** in some features that have been turned into a '\N' value. Each row describes one driver's Grand Prix result and the constructor they raced for. Join on `raceId` to add the event details from `races_df`, on `driverId` for driver details from `drivers_df`, or on `constructorId` for team details from `constructors_df`. The same race-and-driver keys also connect these outcomes to qualifying, lap-time, and pit-stop records.

In [25]:
seasons_df = pd.read_csv("data/seasons.csv")
seasons_df.head()

,year,url
0,2009,http://en.wikipedia.org/wiki/2009_Formula_One_...
1,2008,http://en.wikipedia.org/wiki/2008_Formula_One_...
2,2007,http://en.wikipedia.org/wiki/2007_Formula_One_...
3,2006,http://en.wikipedia.org/wiki/2006_Formula_One_...
4,2005,http://en.wikipedia.org/wiki/2005_Formula_One_...


In [26]:
seasons_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   year    75 non-null     int64
 1   url     75 non-null     str  
dtypes: int64(1), str(1)
memory usage: 5.2 KB


The `seasons_df` dataset has **2 variables** and **no missing values**. It lists championship years and their corresponding URLs. Its `year` field matches `races_df.year`, making it straightforward to associate each race with its season and group the calendar by year.

In [46]:
sprint_results_df = pd.read_csv("data/sprint_results.csv")
sprint_results_df.head(10)

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,fastestLapTime,statusId
0,1,1061,830,9,33,2,1,1,1,3,17,25:38.426,1538426,14,1:30.013,1
1,2,1061,1,131,44,1,2,2,2,2,17,+1.430,1539856,17,1:29.937,1
2,3,1061,822,131,77,3,3,3,3,1,17,+7.502,1545928,17,1:29.958,1
3,4,1061,844,6,16,4,4,4,4,0,17,+11.278,1549704,16,1:30.163,1
4,5,1061,846,1,4,6,5,5,5,0,17,+24.111,1562537,16,1:30.566,1
5,6,1061,817,1,3,7,6,6,6,0,17,+30.959,1569385,17,1:30.640,1
6,7,1061,4,214,14,11,7,7,7,0,17,+43.527,1581953,17,1:31.773,1
7,8,1061,20,117,5,10,8,8,8,0,17,+44.439,1582865,17,1:31.687,1
8,9,1061,847,3,63,8,9,9,9,0,17,+46.652,1585078,17,1:32.208,1
9,10,1061,839,214,31,13,10,10,10,0,17,+47.395,1585821,16,1:32.183,1


In [28]:
sprint_results_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 360 entries, 0 to 359
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   resultId        360 non-null    int64
 1   raceId          360 non-null    int64
 2   driverId        360 non-null    int64
 3   constructorId   360 non-null    int64
 4   number          360 non-null    int64
 5   grid            360 non-null    int64
 6   position        360 non-null    str  
 7   positionText    360 non-null    str  
 8   positionOrder   360 non-null    int64
 9   points          360 non-null    int64
 10  laps            360 non-null    int64
 11  time            360 non-null    str  
 12  milliseconds    360 non-null    str  
 13  fastestLap      360 non-null    str  
 14  fastestLapTime  360 non-null    str  
 15  statusId        360 non-null    int64
dtypes: int64(10), str(6)
memory usage: 54.3 KB


The `sprint_results_df` dataset has **16 variables** and **no missing values**. It records each driver's sprint outcome, with fields similar to those in the Grand Prix results dataset. Its `raceId`, `driverId`, and `constructorId` fields identify the event, driver, and team, so sprint entries can be related to `races_df`, `drivers_df`, and `constructors_df`; the race-and-driver pair also helps compare sprint performance with `results_df` and `qualifying_df`.

In [47]:
status_df = pd.read_csv("data/status.csv")
status_df.head(10)

,statusId,status
0,1,Finished
1,2,Disqualified
2,3,Accident
3,4,Collision
4,5,Engine
5,6,Gearbox
6,7,Transmission
7,8,Clutch
8,9,Hydraulics
9,10,Electrical


In [30]:
status_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 139 entries, 0 to 138
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   statusId  139 non-null    int64
 1   status    139 non-null    str  
dtypes: int64(1), str(1)
memory usage: 3.5 KB


The `status_df` dataset has **2 variables** and **no missing values**. It maps each `statusId` to a human-readable description, such as a finishing or retirement status. Joining on `statusId` adds that explanation to the corresponding rows in `results_df` and `sprint_results_df`.